In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import kagglehub

# Download latest version
path = kagglehub.dataset_download("eunglaimeng/valentines-day-gift-and-satisfaction-dataset-2025")

print("Path to dataset files:", path)
print(os.listdir(path))
df = pd.read_csv(os.path.join(path, 'valentine_gifts_satisfaction.csv'))

# Drop gift_item for now
categorical_features = ["receiver_gender", "relationship_type", "gift_category"]
numerical_features = ["receiver_age", "months_together", "price"]

# Ensure categorical columns are strings
for col in categorical_features:
    df[col] = df[col].astype(str)

X = df[categorical_features + numerical_features]
y = df["receiver_satisfaction"]

# Preprocessing + pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features), # x by x and turns strings into number. ex 3 category: 001, 010, 100
        ("num", StandardScaler(), numerical_features) #Normalize everything, so doesn't end up with price 500 being more important than age 20.
    ]
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=300, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)
print("MSE:", mean_squared_error(y_test, preds))

100%|██████████| 16.9k/16.9k [00:00<00:00, 21.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/eunglaimeng/valentines-day-gift-and-satisfaction-dataset-2025/versions/3
['receiver_names_emails.csv', 'valentine_gifts_satisfaction.csv']


MSE: 0.7001310116086236


In [2]:
print(pipeline.feature_names_in_)

['receiver_gender' 'relationship_type' 'gift_category' 'receiver_age'
 'months_together' 'price']


In [3]:
def recommend_gift(receiver_gender,
                   receiver_age,
                   relationship_type,
                   months_together,
                   max_price):

    categories = df["gift_category"].unique()
    category_scores = []

    for cat in categories:
        # Filter only items under budget
        sub_df = df[df["price"] <= max_price].copy()
        if sub_df.empty:
            continue

        # Build a test row for this category
        test_row = pd.DataFrame([{
            "receiver_gender": receiver_gender,
            "relationship_type": relationship_type,
            "gift_category": cat,
            "receiver_age": receiver_age,
            "months_together": months_together,
            "price": max_price
        }])

        # Predict satisfaction
        score = pipeline.predict(test_row)[0]
        category_scores.append((cat, score))

    # Sort top 3
    category_scores.sort(key=lambda x: x[1], reverse=True)
    top3 = category_scores[:3]

    # Build plots
    plots = []
    for cat, score in top3:
        items = df[(df["gift_category"] == cat) & (df["price"] <= max_price)]
        item_counts = items["gift_item"].value_counts(normalize=True)
        item_counts = item_counts.head(5)

        fig, ax = plt.subplots(figsize=(7,7))
        fig.patch.set_facecolor('#ffe6f0')
        ax.set_facecolor('#fff0f5')

        if item_counts.empty:
            ax.text(0.5, 0.5, "No items under budget 💔",
                    ha='center', va='center', fontsize=14)
            ax.axis("off")
        else:
            ax.pie(
                item_counts.values,
                labels=item_counts.index,
                autopct="%1.1f%%",
                startangle=90
            )
            ax.set_title(f"{cat} \nPredicted Score: {score:.2f}", fontsize=14)

        plots.append(fig)
        plt.close(fig)

    # Guarantee 3 plots
    while len(plots) < 3:
        fig, ax = plt.subplots(figsize=(7,7))
        ax.axis("off")
        plots.append(fig)
        plt.close(fig)

    return plots[0], plots[1], plots[2]



In [4]:
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="pink", secondary_hue="rose"),
    css="body { background-color: #ffe6f0; }"
) as demo:

    gr.Markdown("""
    # 💖 AI Gift Recommender 💖
    Find the perfect gift for your special someone 💌
    """)

    with gr.Row():
        gender = gr.Dropdown(["male", "female", "non-binary"], value="male", label="Receiver Gender")
        relationship = gr.Dropdown(
            ["Dating", "Married", "First Date", "Engaged", "Complicated"],
            value="Dating",
            label="Relationship Type"
        )

    with gr.Row():
        age = gr.Slider(18, 100, step=1, value=18, label="Receiver Age")
        months = gr.Slider(0, 400, step=1, value=0, label="Months Together")
        price = gr.Slider(0, 1000, step=10, value=0, label="Max Budget (USD)")

    submit = gr.Button("Find Gift 💘")

    with gr.Row():
        plot1 = gr.Plot()
        plot2 = gr.Plot()
        plot3 = gr.Plot()

    submit.click(
        recommend_gift,
        inputs=[gender, age, relationship, months, price],
        outputs=[plot1, plot2, plot3]
    )

demo.launch()



/tmp/ipython-input-3644256337.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipython-input-3644256337.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e07f6e222982b78287.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
